# 02 - Grid, Isochrone Proxies, and Spatial Joins

This notebook builds a 500 m grid over the supplied Shanghai POI extent, computes mode-specific 15-minute proximity counts, and caches the grid-level scores. The assignment asks for four modes: walk, bike, transit, and car. Because no routing API key or routable network is included, the implementation uses transparent Euclidean buffers as a reproducible isochrone proxy.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED = ROOT / "data" / "processed"
print(ROOT)

## Run Grid and Mode-Buffer Scoring

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, str(ROOT / "src" / "02_grid_isochrones.py")], check=True)

## Inspect Grid Coverage

In [ ]:
grid = pd.read_csv(PROCESSED / "grid_cells.csv")
scores = pd.read_csv(PROCESSED / "grid_scores.csv")
summary = json.loads((PROCESSED / "grid_summary.json").read_text(encoding="utf-8"))
print(summary)
display(grid.head())
display(scores.head())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
sample = scores.sample(min(12000, len(scores)), random_state=42)
ax.scatter(sample["lon"], sample["lat"], s=1, alpha=0.25)
ax.set_title("500 m Grid Sample over Shanghai POI Extent")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal", adjustable="box")
plt.show()

## Spatial Method

For each grid centroid and each mode radius, the script uses `scipy.spatial.cKDTree` to count POIs in every baseline and Track B category. Counts are converted into scores from 0 to 1 with an adequacy plus richness transform. This keeps the first nearby amenity meaningful while preserving differences between sparse and dense nightlife areas.